In [1]:
from typing import Annotated, TypedDict
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_tavily import TavilySearch
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from dotenv import load_dotenv
import gradio as gr
from IPython.display import display, Image


c:\Users\Maneesha\Projects\Multi-Agent_Langraph\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv(override=True)
google_api_key=os.getenv("GOOGLE_API_KEY")

In [ ]:
llm=ChatOpenAI(model="llama3.2:1b",api_key="key", base_url="http://localhost:11434/v1")
# llm=ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite",api_key=google_api_key)
#memory=MemorySaver()

DB_path="memory.db"
conn=sqlite3.connect(DB_path, check_same_thread=False)
sql_memory=SqliteSaver(conn)


In [4]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

In [5]:
graph_builder=StateGraph(State)

In [6]:
tavily_search=TavilySearch(max_results=2)

@tool
def web_search(query: str):
    '''search the web'''
    return tavily_search.invoke(query)

In [7]:
tools=[web_search]
llm_with_tools=llm.bind_tools(tools)

In [8]:
def chatbot(state: State):
    response=llm_with_tools.invoke(state["messages"])
    return { "messages" : [response]}

In [9]:
graph_builder.add_node("chat_bot",chatbot)
graph_builder.add_node("tools", ToolNode(tools))
graph_builder.add_edge(START, "chat_bot")
graph_builder.add_conditional_edges("chat_bot",tools_condition)
graph_builder.add_edge("tools","chat_bot")


In [10]:
graph=graph_builder.compile(checkpointer=sql_memory)
config = {"configurable": {"thread_id": "1"}}

In [11]:
def chat_llm(user_msg: str, history):
    messages=[HumanMessage(content=user_msg)]
    state={"messages": messages}
    response=graph.invoke(state,config=config)
    return response["messages"][-1].content

In [14]:
gr.ChatInterface(fn=chat_llm).launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [13]:
graph.get_state(config)

StateSnapshot(values={'messages': [HumanMessage(content='hi my name is maneesha', additional_kwargs={}, response_metadata={}, id='00728f80-85e4-4c7d-9e4a-04afc1abd155'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 146, 'total_tokens': 170, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'llama3.2:1b', 'system_fingerprint': 'fp_ollama', 'id': 'chatcmpl-510', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e2ab3-605e-7b23-9253-d6ad30821bc5-0', tool_calls=[{'name': 'web_search', 'args': {'search': '{"query":{"type":"string"}'}, 'id': 'call_aglo9sno', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 146, 'output_tokens': 24, 'total_tokens': 170, 'input_token_details': {}, 'output_token_details': {}}), ToolMessage(content='Error invoking tool \'web_search\' with kwargs {\'search\': \'{"query":{"